# Firestore Database Setup

This notebook demonstrates how to:
1. Enable the Firestore API
2. Create a Firestore database in Native Mode
3. Verify the setup with basic operations

**Firestore Mode**: Native Mode - Document database with real-time updates and offline support

**Location**: Single region (us-central1) for this demo

**Estimated Time**: 5-8 minutes (including database provisioning which takes 2-3 minutes)

**Required Permissions**:
- `roles/datastore.owner` (to create and manage Firestore databases)

**Cost Considerations**:

Firestore offers a generous free tier:
- 1 GB storage
- 50,000 document reads per day
- 20,000 document writes per day
- 20,000 document deletes per day

This basic setup will remain well within free tier limits.

## Section 1: Configuration & Authentication

In [ ]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
DATABASE_ID = "(default)"  # Use default Firestore database

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Database ID: {DATABASE_ID}")

In [ ]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

In [ ]:
# Check required IAM permissions
print("🔐 Checking required IAM permissions...\n")
print("This notebook requires the following permissions:\n")
print("   1️⃣  Firestore:")
print("      - roles/datastore.owner (to create and manage Firestore)\n")

# Try to get user email
try:
    if hasattr(credentials, 'service_account_email'):
        user_email = credentials.service_account_email
    elif hasattr(credentials, '_service_account_email'):
        user_email = credentials._service_account_email
    else:
        user_email = "(Unable to detect email from credentials)"
except:
    user_email = "(Unable to detect email from credentials)"

print(f"   Your account: {user_email}\n")
print("💡 To grant these permissions, run these commands in your terminal:\n")
print("   # Grant Datastore Owner")
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/datastore.owner'\n")

In [ ]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = [
    "google-cloud-firestore"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

In [ ]:
# Import libraries
import time
import requests
from google.auth.transport.requests import Request
from google.cloud import firestore

# Refresh credentials for REST API calls
credentials.refresh(Request())

print("✅ Libraries imported and credentials refreshed")

## Section 2: Enable Required APIs

Enable the Firestore API for the project.

In [ ]:
# Check if Firestore API is enabled
print("🔍 Checking Firestore API status...")

api_url = f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/firestore.googleapis.com"
headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

response = requests.get(api_url, headers=headers)

if response.status_code == 200:
    service_info = response.json()
    state = service_info.get('state', 'UNKNOWN')
    
    if state == 'ENABLED':
        print("✅ Firestore API is already enabled")
        API_ENABLED = True
    else:
        print(f"ℹ️  Firestore API state: {state}")
        API_ENABLED = False
else:
    print(f"⚠️  Could not check API status: {response.status_code}")
    API_ENABLED = False

In [ ]:
# Enable Firestore API if needed
if not API_ENABLED:
    print("🚀 Enabling Firestore API...")
    
    enable_url = f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/firestore.googleapis.com:enable"
    
    response = requests.post(enable_url, headers=headers)
    
    if response.status_code == 200:
        print("✅ Firestore API enabled successfully")
        print("⏳ Waiting for API to become fully available...")
        time.sleep(10)  # Wait for API to propagate
        API_ENABLED = True
    else:
        print(f"❌ Failed to enable Firestore API: {response.status_code}")
        print(response.text)
        print("\n💡 Try enabling it manually:")
        print(f"   gcloud services enable firestore.googleapis.com --project={PROJECT_ID}")
else:
    print("ℹ️  Skipping API enablement - already enabled")

## Section 3: Create Firestore Database

This section creates a Firestore database in Native Mode using the Firestore Admin API.

**Database Configuration:**
- Type: Native Mode (for real-time and mobile applications)
- Location: us-central1 (single region)
- Database ID: (default)

**Note**: Database creation takes approximately 2-3 minutes.

In [ ]:
# Check if Firestore database already exists
print("🔍 Checking if Firestore database exists...")

# For the default database, we need to check using Firestore Admin API
db_url = f"https://firestore.googleapis.com/v1/projects/{PROJECT_ID}/databases/{DATABASE_ID}"

response = requests.get(db_url, headers=headers)

if response.status_code == 200:
    db_info = response.json()
    print(f"✅ Firestore database '{DATABASE_ID}' already exists")
    print(f"   Name: {db_info.get('name', 'N/A')}")
    print(f"   Type: {db_info.get('type', 'N/A')}")
    print(f"   Location: {db_info.get('locationId', 'N/A')}")
    DATABASE_EXISTS = True
elif response.status_code == 404:
    print(f"ℹ️  Firestore database '{DATABASE_ID}' does not exist yet")
    DATABASE_EXISTS = False
else:
    print(f"⚠️  Error checking database: {response.status_code}")
    print(response.text)
    DATABASE_EXISTS = False

In [ ]:
# Create Firestore database if it doesn't exist
if not DATABASE_EXISTS:
    print(f"🚀 Creating Firestore database '{DATABASE_ID}'...")
    print("⏱️  This will take approximately 2-3 minutes")
    
    create_url = f"https://firestore.googleapis.com/v1/projects/{PROJECT_ID}/databases"
    
    database_config = {
        "name": f"projects/{PROJECT_ID}/databases/{DATABASE_ID}",
        "locationId": REGION,
        "type": "FIRESTORE_NATIVE"
    }
    
    # Note: databaseId parameter goes as query string
    create_url_with_id = f"{create_url}?databaseId={DATABASE_ID}"
    
    response = requests.post(create_url_with_id, headers=headers, json=database_config)
    
    if response.status_code in [200, 201]:
        print("✅ Database creation started")
        operation = response.json()
        operation_name = operation.get('name')
        
        if operation_name:
            print(f"   Operation: {operation_name}")
            
            # Wait for operation to complete
            print("\n⏳ Waiting for database creation to complete...")
            max_wait_time = 300  # 5 minutes
            start_time = time.time()
            
            while time.time() - start_time < max_wait_time:
                op_url = f"https://firestore.googleapis.com/v1/{operation_name}"
                op_response = requests.get(op_url, headers=headers)
                
                if op_response.status_code == 200:
                    op_data = op_response.json()
                    
                    if op_data.get('done', False):
                        if 'error' in op_data:
                            print(f"\n❌ Database creation failed: {op_data['error']}")
                            break
                        else:
                            elapsed = int(time.time() - start_time)
                            print(f"\n✅ Database created successfully! (took {elapsed}s)")
                            DATABASE_EXISTS = True
                            break
                    else:
                        elapsed = int(time.time() - start_time)
                        print(f"   Waiting... {elapsed}s elapsed", end='\r')
                
                time.sleep(5)  # Check every 5 seconds
            
            if not DATABASE_EXISTS:
                elapsed = int(time.time() - start_time)
                print(f"\n⚠️  Database creation timed out after {elapsed}s")
                print("   Please check the GCP Console to verify database status.")
        else:
            # Response might be the database itself (immediate creation)
            print("✅ Database created successfully!")
            DATABASE_EXISTS = True
    else:
        print(f"❌ Failed to create database: {response.status_code}")
        print(response.text)
        print("\n💡 You may need to create the database using the console:")
        print(f"   https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")
else:
    print("ℹ️  Skipping database creation - database already exists")

In [ ]:
# Verify database status
if DATABASE_EXISTS:
    print("\n🔍 Verifying database status...")
    
    response = requests.get(db_url, headers=headers)
    
    if response.status_code == 200:
        db_info = response.json()
        print("\n📊 Database Information:")
        print(f"   Name: {db_info.get('name', 'N/A')}")
        print(f"   Type: {db_info.get('type', 'N/A')}")
        print(f"   Location: {db_info.get('locationId', 'N/A')}")
        print(f"\n🔗 View in Console:")
        print(f"   https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")
    else:
        print(f"⚠️  Could not verify database: {response.status_code}")
        print(response.text)

## Section 4: Verify Firestore Setup

Test the Firestore setup with basic create, read, and delete operations.

In [ ]:
# Initialize Firestore client
print("🔌 Initializing Firestore client...")

try:
    db = firestore.Client(project=PROJECT_ID, database=DATABASE_ID)
    print("✅ Firestore client initialized successfully")
    print(f"   Project: {PROJECT_ID}")
    print(f"   Database: {DATABASE_ID}")
except Exception as e:
    print(f"❌ Failed to initialize Firestore client: {e}")
    print("\n💡 Make sure the Firestore API is enabled and you have proper permissions.")
    raise

In [ ]:
# Test: Create a document
print("\n📝 Testing: Create a test document...")

test_collection = "test_setup"
test_doc_id = "verification_doc"

test_data = {
    "message": "Hello from Firestore!",
    "timestamp": firestore.SERVER_TIMESTAMP,
    "test_number": 42,
    "test_boolean": True,
    "nested_data": {
        "field1": "value1",
        "field2": "value2"
    }
}

try:
    doc_ref = db.collection(test_collection).document(test_doc_id)
    doc_ref.set(test_data)
    print(f"✅ Document created: {test_collection}/{test_doc_id}")
except Exception as e:
    print(f"❌ Failed to create document: {e}")
    raise

In [ ]:
# Test: Read the document
print("\n📖 Testing: Read the test document...")

try:
    doc_ref = db.collection(test_collection).document(test_doc_id)
    doc = doc_ref.get()
    
    if doc.exists:
        print(f"✅ Document retrieved successfully")
        print(f"\n   Document data:")
        for key, value in doc.to_dict().items():
            print(f"     {key}: {value}")
    else:
        print(f"❌ Document does not exist")
except Exception as e:
    print(f"❌ Failed to read document: {e}")
    raise

In [ ]:
# Test: Query the collection
print("\n🔍 Testing: Query the collection...")

try:
    docs = db.collection(test_collection).limit(5).stream()
    
    doc_count = 0
    for doc in docs:
        doc_count += 1
        print(f"   Document {doc.id}: {doc.to_dict().get('message', 'N/A')}")
    
    print(f"\n✅ Query successful - found {doc_count} document(s)")
except Exception as e:
    print(f"❌ Failed to query collection: {e}")
    raise

In [ ]:
# Test: Delete the test document
print("\n🗑️  Testing: Delete the test document...")

try:
    doc_ref = db.collection(test_collection).document(test_doc_id)
    doc_ref.delete()
    print(f"✅ Document deleted: {test_collection}/{test_doc_id}")
    
    # Verify deletion
    doc = doc_ref.get()
    if not doc.exists:
        print("✅ Deletion verified - document no longer exists")
    else:
        print("⚠️  Document still exists after deletion attempt")
except Exception as e:
    print(f"❌ Failed to delete document: {e}")
    raise

In [ ]:
# Final verification
print("\n✅ All tests passed! Firestore is set up and working correctly.\n")

## Section 5: Summary & Next Steps

Congratulations! You've successfully set up Firestore in your GCP project.

In [ ]:
# Display summary
print("="*60)
print("🎉 FIRESTORE SETUP COMPLETE")
print("="*60)
print("\n📊 Resources Created:\n")
print(f"   ✅ Firestore Database: {DATABASE_ID}")
print(f"   ✅ Location: {REGION}")
print(f"   ✅ Type: FIRESTORE_NATIVE")
print("\n🔗 Quick Links:\n")
print(f"   📂 Firestore Console:")
print(f"      https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")
print(f"\n   📊 Firestore Data:")
print(f"      https://console.cloud.google.com/firestore/data?project={PROJECT_ID}")
print(f"\n   📈 Firestore Usage:")
print(f"      https://console.cloud.google.com/firestore/usage?project={PROJECT_ID}")
print("\n💡 Next Steps:\n")
print("   1. Start adding collections and documents to your database")
print("   2. Explore Firestore features like real-time listeners")
print("   3. Set up security rules for production use")
print("   4. Consider integrating with Firebase for mobile apps")
print("\n📚 Useful Documentation:\n")
print("   - Firestore Quickstart: https://firebase.google.com/docs/firestore/quickstart")
print("   - Data Model: https://firebase.google.com/docs/firestore/data-model")
print("   - Security Rules: https://firebase.google.com/docs/firestore/security/get-started")
print("   - Python Client: https://cloud.google.com/python/docs/reference/firestore/latest")
print("\n💰 Cost Information:\n")
print("   Firestore Free Tier (per day):")
print("   - 1 GB storage")
print("   - 50,000 document reads")
print("   - 20,000 document writes")
print("   - 20,000 document deletes")
print("\n   Full pricing: https://firebase.google.com/pricing")
print("\n⚠️  Cleanup Instructions:\n")
print("   To delete the Firestore database:")
print("   1. Go to: https://console.cloud.google.com/firestore/databases")
print("   2. Select your database")
print("   3. Click 'Delete database'")
print("\n   Or use gcloud CLI:")
print(f"   gcloud firestore databases delete {DATABASE_ID} --project={PROJECT_ID}")
print("\n" + "="*60)

## Example: Basic Firestore Operations

Here are some common Firestore operations you can try:

In [ ]:
# Example: Create a collection with multiple documents
print("📝 Example: Creating sample documents...\n")

# Sample data
users = [
    {"name": "Alice", "age": 30, "city": "New York"},
    {"name": "Bob", "age": 25, "city": "San Francisco"},
    {"name": "Charlie", "age": 35, "city": "Boston"}
]

# Create documents
for user in users:
    doc_ref = db.collection("example_users").document()
    doc_ref.set(user)
    print(f"   Created user: {user['name']}")

print("\n✅ Sample documents created in 'example_users' collection")

In [ ]:
# Example: Query with filters
print("\n🔍 Example: Querying users older than 28...\n")

users_ref = db.collection("example_users")
query = users_ref.where("age", ">", 28).order_by("age")

results = query.stream()
for doc in results:
    data = doc.to_dict()
    print(f"   {data['name']}: {data['age']} years old, {data['city']}")

print("\n✅ Query completed")

In [ ]:
# Example: Update a document
print("\n✏️  Example: Updating a document...\n")

# Find Bob's document
users_ref = db.collection("example_users")
query = users_ref.where("name", "==", "Bob").limit(1)
docs = list(query.stream())

if docs:
    doc_ref = docs[0].reference
    doc_ref.update({"age": 26, "city": "Seattle"})
    print("   Updated Bob's information")
    
    # Read it back
    updated_doc = doc_ref.get()
    data = updated_doc.to_dict()
    print(f"   New data: {data['name']}, {data['age']} years old, {data['city']}")

print("\n✅ Update completed")

In [ ]:
# Example: Batch operations
print("\n🔄 Example: Batch write operations...\n")

batch = db.batch()

# Create references
ref1 = db.collection("example_batch").document("doc1")
ref2 = db.collection("example_batch").document("doc2")
ref3 = db.collection("example_batch").document("doc3")

# Add operations to batch
batch.set(ref1, {"content": "First document", "order": 1})
batch.set(ref2, {"content": "Second document", "order": 2})
batch.set(ref3, {"content": "Third document", "order": 3})

# Commit batch
batch.commit()

print("   Created 3 documents in a single batch operation")
print("\n✅ Batch operation completed")

In [ ]:
# Cleanup example data (optional)
print("\n🗑️  Cleaning up example data...\n")

# Delete example_users collection
users_ref = db.collection("example_users")
delete_count = 0
for doc in users_ref.stream():
    doc.reference.delete()
    delete_count += 1

print(f"   Deleted {delete_count} documents from example_users")

# Delete example_batch collection
batch_ref = db.collection("example_batch")
delete_count = 0
for doc in batch_ref.stream():
    doc.reference.delete()
    delete_count += 1

print(f"   Deleted {delete_count} documents from example_batch")
print("\n✅ Example data cleaned up")